In [3]:
import socket
import json

def send_request(option, input_data=None):
    payload = {"option": option}
    if input_data:
        payload["input_data"] = input_data
        
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.connect(("socket.cryptohack.org", 13372))
    
    rfile = s.makefile("rw", encoding="utf-8")
    rfile.readline()
    
    rfile.write(json.dumps(payload) + "\n")
    rfile.flush()
    
    response_line = rfile.readline()
    s.close()
    return json.loads(response_line)

def pwn():
    while True:
        try:
            res1 = send_request("get_flag")
            if "error" in res1:
                continue
                
            c_flag = bytes.fromhex(res1["encrypted_flag"])
            length = len(c_flag)
            known_plaintext = b"A" * length
            
            res2 = send_request("encrypt_data", input_data=known_plaintext.hex())
            c_data = bytes.fromhex(res2["encrypted_data"])
            
            flag = bytes([f ^ d ^ p for f, d, p in zip(c_flag, c_data, known_plaintext)])
            print(flag.decode("utf-8"))
            break
        except (UnicodeDecodeError, KeyError):
            continue

if __name__ == "__main__":
    pwn()

crypto{t00_f4st_t00_furi0u5}
